# News Category Prediction — Approche LLM avec Ollama

Ce notebook reprend le même dataset et les mêmes objectifs que le notebook de référence (classification d'articles HuffPost en catégories), mais remplace le pipeline NLP classique (TF-IDF + Logistic Regression) par une **prédiction via un LLM local** servi par **Ollama**.

### Approches testées
1. **Zero-shot** : on donne la liste des catégories et le texte, le LLM prédit.
2. **Few-shot** : on ajoute quelques exemples dans le prompt pour guider le modèle.
3. **Évaluation** : Accuracy et MRR@k, identiques au notebook de référence.

### Prérequis
- [Ollama](https://ollama.ai) installé et lancé (`ollama serve`)
- Un modèle téléchargé, par ex. `ollama pull llama3.1:8b` ou `mistral`

---
## 1. Installation & Imports

In [17]:
import pandas as pd
import numpy as np
import json
import re
import time
import logging
from tqdm import tqdm
from sklearn.model_selection import train_test_split

import ollama

logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

---
## 2. Configuration Ollama

In [18]:
# ---- Configuration ----
OLLAMA_MODEL = "llama3.1:8b"
EVAL_SAMPLE_SIZE = 200
TOP_K = 3

# Vérifier que Ollama est joignable
try:
    response = ollama.list()
    print("Ollama est connecté. Modèles disponibles :")
    # Afficher la structure pour debug
    print(type(response))
    print(response)
except Exception as e:
    print(f"Erreur de connexion à Ollama : {e}")

2026-04-14 22:57:16,960 : INFO : HTTP Request: GET http://127.0.0.1:11434/api/tags "HTTP/1.1 200 OK"


Ollama est connecté. Modèles disponibles :
<class 'ollama._types.ListResponse'>
models=[Model(model='llama3.1:8b', modified_at=datetime.datetime(2026, 4, 14, 22, 26, 0, 102035, tzinfo=TzInfo(7200)), digest='46e0c10c039e019119339687c3c1757cc81b9da49709a3b3924863ba87ca666e', size=4920753328, details=ModelDetails(parent_model='', format='gguf', family='llama', families=['llama'], parameter_size='8.0B', quantization_level='Q4_K_M'))]


---
## 3. Chargement des données

In [19]:
df = pd.read_json(r"C:\Users\jfurs\Pythonn\OpenClassrooms\DS\P8\high_end_ollama_solution\data\news_category_dataset.json", lines=True)
print(f"Nombre d'articles : {len(df)}")
print(f"Colonnes : {list(df.columns)}")
df.head(3)

Nombre d'articles : 124989
Colonnes : ['short_description', 'headline', 'date', 'link', 'authors', 'category']


,short_description,headline,date,link,authors,category
0,She left her husband. He killed their children...,There Were 2 Mass Shootings In Texas Last Week...,2018-05-26,https://www.huffingtonpost.com/entry/texas-ama...,Melissa Jeltsen,CRIME
1,Of course it has a song.,Will Smith Joins Diplo And Nicky Jam For The 2...,2018-05-26,https://www.huffingtonpost.com/entry/will-smit...,Andy McDonald,ENTERTAINMENT
2,The actor and his longtime girlfriend Anna Ebe...,Hugh Grant Marries For The First Time At Age 57,2018-05-26,https://www.huffingtonpost.com/entry/hugh-gran...,Ron Dicker,ENTERTAINMENT


In [20]:
# Liste des catégories du dataset
CATEGORIES = sorted(df['category'].unique().tolist())
print(f"Nombre de catégories : {len(CATEGORIES)}")
print(CATEGORIES)

Nombre de catégories : 31
['ARTS', 'ARTS & CULTURE', 'BLACK VOICES', 'BUSINESS', 'COLLEGE', 'COMEDY', 'CRIME', 'EDUCATION', 'ENTERTAINMENT', 'FIFTY', 'GOOD NEWS', 'GREEN', 'HEALTHY LIVING', 'IMPACT', 'LATINO VOICES', 'MEDIA', 'PARENTS', 'POLITICS', 'QUEER VOICES', 'RELIGION', 'SCIENCE', 'SPORTS', 'STYLE', 'TASTE', 'TECH', 'THE WORLDPOST', 'TRAVEL', 'WEIRD NEWS', 'WOMEN', 'WORLD NEWS', 'WORLDPOST']


### Préparation du texte (identique au notebook de référence)

In [21]:
def tokenize_url(url: str):
    url = url.replace("https://www.huffingtonpost.com/entry/", "")
    url = re.sub(r"(\W|_)+", " ", url)
    return url

df['tokenized_url'] = df['link'].apply(tokenize_url)

# Texte combiné : description + headline
df['text'] = df['headline'] + '. ' + df['short_description']
df['text'] = df['text'].fillna('')

df[['category', 'headline', 'short_description', 'text']].sample(5)

,category,headline,short_description,text
116630,RELIGION,This Bible Is Designed To Be An Enjoyable Read,,This Bible Is Designed To Be An Enjoyable Read.
15351,POLITICS,Our System Is Rigged,Our system weights some votes more than others...,Our System Is Rigged. Our system weights some ...
69342,STYLE,Axe Wants To Shed Its Douchey Reputation And E...,"Goodbye, ""Axe effect.""",Axe Wants To Shed Its Douchey Reputation And E...
53284,POLITICS,"SATIRE: Trump, The Pop Musical; Or, A Few Su...",The Trump campaign needs a theme song designed...,"SATIRE: Trump, The Pop Musical; Or, A Few Su..."
46133,POLITICS,Maryland Man Charged With Theft Of Classified ...,"Harold Thomas Martin, 51, worked as a contract...",Maryland Man Charged With Theft Of Classified ...


### Split train / test

In [22]:
training_data, testing_data = train_test_split(df, random_state=2000)
print(f"Train : {len(training_data)} | Test : {len(testing_data)}")

# Sous-échantillonnage du test pour l'évaluation LLM (coût en temps)
eval_data = testing_data.sample(n=min(EVAL_SAMPLE_SIZE, len(testing_data)), random_state=42).reset_index(drop=True)
print(f"Échantillon d'évaluation : {len(eval_data)} articles")

Train : 93741 | Test : 31248
Échantillon d'évaluation : 200 articles


---
## 4. Fonctions de prédiction via Ollama

In [23]:
def build_zero_shot_prompt(text: str, categories: list, top_k: int = 3) -> str:
    """Construit un prompt zero-shot pour la classification."""
    cats_str = ", ".join(categories)
    prompt = f"""You are a news article classifier. Your task is to assign the most appropriate category to a news article.

CATEGORIES:
{cats_str}

ARTICLE:
\"\"\"
{text}
\"\"\"

Return EXACTLY {top_k} categories from the list above, ranked from most likely to least likely.
Respond ONLY with a JSON array of strings, nothing else. Example: ["POLITICS", "BUSINESS", "CRIME"]
"""
    return prompt


def build_few_shot_prompt(text: str, categories: list, examples: list, top_k: int = 3) -> str:
    """Construit un prompt few-shot avec des exemples."""
    cats_str = ", ".join(categories)
    
    examples_str = ""
    for ex in examples:
        examples_str += f'Article: "{ex["text"]}"\nCategory: {ex["category"]}\n\n'
    
    prompt = f"""You are a news article classifier. Your task is to assign the most appropriate category to a news article.

CATEGORIES:
{cats_str}

Here are some examples:

{examples_str}
Now classify this article:

ARTICLE:
\"\"\"
{text}
\"\"\"

Return EXACTLY {top_k} categories from the list above, ranked from most likely to least likely.
Respond ONLY with a JSON array of strings, nothing else. Example: ["POLITICS", "BUSINESS", "CRIME"]
"""
    return prompt


print("Exemple de prompt zero-shot :")
print(build_zero_shot_prompt("Biden announces new climate policy for 2025", CATEGORIES[:5], top_k=3))

Exemple de prompt zero-shot :
You are a news article classifier. Your task is to assign the most appropriate category to a news article.

CATEGORIES:
ARTS, ARTS & CULTURE, BLACK VOICES, BUSINESS, COLLEGE

ARTICLE:
"""
Biden announces new climate policy for 2025
"""

Return EXACTLY 3 categories from the list above, ranked from most likely to least likely.
Respond ONLY with a JSON array of strings, nothing else. Example: ["POLITICS", "BUSINESS", "CRIME"]



In [24]:
def parse_llm_response(response_text: str, valid_categories: list, top_k: int = 3) -> list:
    """
    Parse la réponse du LLM pour en extraire les catégories prédites.
    Gère les formats variés que le LLM peut retourner.
    """
    text = response_text.strip()
    
    # Tenter le parsing JSON direct
    try:
        # Extraire le JSON s'il est entouré de texte
        json_match = re.search(r'\[.*?\]', text, re.DOTALL)
        if json_match:
            parsed = json.loads(json_match.group())
            # Normaliser en majuscules et filtrer les catégories valides
            preds = [p.strip().upper() for p in parsed if isinstance(p, str)]
            preds = [p for p in preds if p in valid_categories]
            if preds:
                return preds[:top_k]
    except (json.JSONDecodeError, AttributeError):
        pass
    
    # Fallback : chercher les catégories mentionnées dans le texte
    text_upper = text.upper()
    found = [cat for cat in valid_categories if cat in text_upper]
    if found:
        return found[:top_k]
    
    # Dernier recours : retourner une liste vide
    return []


# Test du parser
test_responses = [
    '["POLITICS", "CRIME", "BUSINESS"]',
    'The article is about POLITICS and CRIME.',
    'Here are my predictions:\n["ENTERTAINMENT", "COMEDY"]',
]
for resp in test_responses:
    print(f"  Input:  {resp!r}")
    print(f"  Parsed: {parse_llm_response(resp, CATEGORIES)}")
    print()

  Input:  '["POLITICS", "CRIME", "BUSINESS"]'
  Parsed: ['POLITICS', 'CRIME', 'BUSINESS']

  Input:  'The article is about POLITICS and CRIME.'
  Parsed: ['CRIME', 'POLITICS']

  Input:  'Here are my predictions:\n["ENTERTAINMENT", "COMEDY"]'
  Parsed: ['ENTERTAINMENT', 'COMEDY']



In [25]:
def predict_with_ollama(text: str, categories: list, model: str = OLLAMA_MODEL,
                         top_k: int = 3, mode: str = "zero_shot",
                         examples: list = None) -> list:
    """
    Appelle Ollama pour prédire les catégories d'un texte.
    
    Args:
        text: le texte de l'article
        categories: liste des catégories possibles
        model: nom du modèle Ollama
        top_k: nombre de catégories à retourner
        mode: 'zero_shot' ou 'few_shot'
        examples: liste de dicts {text, category} pour le few-shot
    
    Returns:
        Liste de catégories prédites (top_k)
    """
    if mode == "few_shot" and examples:
        prompt = build_few_shot_prompt(text, categories, examples, top_k)
    else:
        prompt = build_zero_shot_prompt(text, categories, top_k)
    
    try:
        response = ollama.chat(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            options={
                "temperature": 0.1,  # Basse température pour des prédictions stables
                "num_predict": 100,  # On n'a besoin que de quelques tokens
            }
        )
        response_text = response['message']['content']
        return parse_llm_response(response_text, categories, top_k)
    
    except Exception as e:
        logging.error(f"Erreur Ollama : {e}")
        return []

### Test rapide sur un article

In [ ]:
# Test sur un article du dataset
sample = eval_data.iloc[0]
print(f"Texte  : {sample['text'][:200]}...")
print(f"Vrai   : {sample['category']}")

pred = predict_with_ollama(sample['text'], CATEGORIES, top_k=TOP_K)
print(f"Prédit : {pred}")

Texte  : Trump Admin Plans To Impose 20 Percent Duties On Canadian Softwood Lumber. The move escalates a long-running trade dispute between the two countries....
Vrai   : POLITICS


---
## 5. Fonctions d'évaluation (reprises du notebook de référence)

In [ ]:
def _reciprocal_rank(true_labels: list, machine_preds: list):
    """Compute the reciprocal rank at cutoff k"""
    tp_pos_list = [(idx + 1) for idx, r in enumerate(machine_preds) if r in true_labels]
    rr = 0
    if len(tp_pos_list) > 0:
        first_pos_list = tp_pos_list[0]
        rr = 1 / float(first_pos_list)
    return rr


def compute_mrr_at_k(items: list):
    """Compute the MRR (average RR) at cutoff k"""
    rr_total = 0
    for item in items:
        rr_at_k = _reciprocal_rank(item[0], item[1])
        rr_total += rr_at_k
    mrr = rr_total / float(len(items))
    return mrr


def compute_accuracy(eval_items: list):
    """Accuracy : le vrai label est-il dans les top-k prédictions ?"""
    correct = 0
    for item in eval_items:
        true_pred = item[0]
        machine_pred = set(item[1])
        for cat in true_pred:
            if cat in machine_pred:
                correct += 1
                break
    accuracy = correct / float(len(eval_items))
    return accuracy


def collect_preds(y_true: list, y_preds: list):
    return [[[y_true[idx]], pred] for idx, pred in enumerate(y_preds)]

---
## 6. Évaluation Zero-Shot

In [ ]:
def run_evaluation(eval_df: pd.DataFrame, categories: list, model: str = OLLAMA_MODEL,
                   top_k: int = 3, mode: str = "zero_shot", examples: list = None):
    """
    Lance la prédiction LLM sur un DataFrame et calcule les métriques.
    """
    all_preds = []
    y_true = eval_df['category'].values.tolist()
    
    start_time = time.time()
    
    for idx, row in tqdm(eval_df.iterrows(), total=len(eval_df), desc=f"Prédiction ({mode})"):
        preds = predict_with_ollama(
            text=row['text'],
            categories=categories,
            model=model,
            top_k=top_k,
            mode=mode,
            examples=examples
        )
        all_preds.append(preds)
    
    elapsed = time.time() - start_time
    
    # Calcul des métriques
    eval_items = collect_preds(y_true, all_preds)
    accuracy = compute_accuracy(eval_items)
    mrr = compute_mrr_at_k(eval_items)
    
    # Accuracy top-1 (la première prédiction est-elle correcte ?)
    top1_correct = sum(1 for true, pred in zip(y_true, all_preds) if pred and pred[0] == true)
    top1_acc = top1_correct / len(y_true)
    
    # Taux d'échec (le LLM n'a retourné aucune catégorie valide)
    empty_preds = sum(1 for p in all_preds if len(p) == 0)
    
    results = {
        "mode": mode,
        "model": model,
        "n_samples": len(eval_df),
        "top_k": top_k,
        "accuracy_at_k": accuracy,
        "top1_accuracy": top1_acc,
        "mrr_at_k": mrr,
        "empty_predictions": empty_preds,
        "time_seconds": round(elapsed, 1),
        "seconds_per_article": round(elapsed / len(eval_df), 2),
    }
    
    return results, all_preds


print("Fonction d'évaluation prête.")

Fonction d'évaluation prête.


In [ ]:
# ---- Évaluation Zero-Shot ----
logging.info(f"Lancement évaluation zero-shot sur {len(eval_data)} articles...")

results_zs, preds_zs = run_evaluation(
    eval_df=eval_data,
    categories=CATEGORIES,
    model=OLLAMA_MODEL,
    top_k=TOP_K,
    mode="zero_shot"
)

print("\n=== Résultats Zero-Shot ===")
for k, v in results_zs.items():
    print(f"  {k:25s} : {v}")

---
## 7. Évaluation Few-Shot

In [ ]:
# Sélection d'exemples pour le few-shot (1 par catégorie, depuis le train)
# On prend un échantillon stratifié pour couvrir un maximum de catégories
few_shot_examples = []

for cat in CATEGORIES:
    cat_samples = training_data[training_data['category'] == cat]
    if len(cat_samples) > 0:
        sample = cat_samples.sample(1, random_state=42).iloc[0]
        few_shot_examples.append({
            "text": sample['text'][:200],  # Tronquer pour limiter la taille du prompt
            "category": cat
        })

print(f"{len(few_shot_examples)} exemples few-shot sélectionnés")
print("\nAperçu (3 premiers) :")
for ex in few_shot_examples[:3]:
    print(f"  [{ex['category']}] {ex['text'][:80]}...")

In [ ]:
# ---- Évaluation Few-Shot ----
# Note : avec ~40 catégories, le prompt few-shot complet peut être long.
# On peut limiter à un sous-ensemble d'exemples si le contexte du modèle est petit.

# Option : réduire à 10 exemples diversifiés si le modèle a un contexte limité
MAX_FEW_SHOT_EXAMPLES = 10
examples_subset = few_shot_examples[:MAX_FEW_SHOT_EXAMPLES]

logging.info(f"Lancement évaluation few-shot ({len(examples_subset)} exemples) sur {len(eval_data)} articles...")

results_fs, preds_fs = run_evaluation(
    eval_df=eval_data,
    categories=CATEGORIES,
    model=OLLAMA_MODEL,
    top_k=TOP_K,
    mode="few_shot",
    examples=examples_subset
)

print("\n=== Résultats Few-Shot ===")
for k, v in results_fs.items():
    print(f"  {k:25s} : {v}")

---
## 8. Comparaison des résultats

In [ ]:
df_results = pd.DataFrame([results_zs, results_fs])
df_results = df_results[['mode', 'model', 'n_samples', 'top_k', 'top1_accuracy',
                          'accuracy_at_k', 'mrr_at_k', 'empty_predictions',
                          'time_seconds', 'seconds_per_article']]
df_results

### Analyse des erreurs

In [ ]:
# Analyse détaillée des prédictions zero-shot
eval_data_analysis = eval_data.copy()
eval_data_analysis['preds_zero_shot'] = preds_zs
eval_data_analysis['preds_few_shot'] = preds_fs
eval_data_analysis['correct_zs'] = eval_data_analysis.apply(
    lambda r: r['category'] in r['preds_zero_shot'], axis=1
)
eval_data_analysis['correct_fs'] = eval_data_analysis.apply(
    lambda r: r['category'] in r['preds_few_shot'], axis=1
)

# Accuracy par catégorie (zero-shot)
cat_accuracy = eval_data_analysis.groupby('category')['correct_zs'].mean().sort_values(ascending=False)
print("Accuracy par catégorie (zero-shot, top-k) :")
print(cat_accuracy.to_string())

In [ ]:
# Exemples d'erreurs
errors = eval_data_analysis[~eval_data_analysis['correct_zs']].head(10)
print("Exemples d'erreurs (zero-shot) :\n")
for _, row in errors.iterrows():
    print(f"  Texte  : {row['text'][:120]}...")
    print(f"  Vrai   : {row['category']}")
    print(f"  Prédit : {row['preds_zero_shot']}")
    print()

---
## 9. Prédictions sur articles CNN (non vus) — comme le notebook de référence

In [ ]:
cnn_articles = [
    {
        "source": "https://www.cnn.com/2019/07/19/politics/george-nader-child-porn-sex-charges/",
        "text": "George Aref Nader, who was a key witness in special counsel Robert Mueller's Russia investigation, faces new charges of transporting a minor with intent to engage in criminal sexual activity and child pornography"
    },
    {
        "source": "https://www.cnn.com/2019/07/18/entertainment/khloe-kardashian-true-thompson-video-trnd/",
        "text": "True Thompson makes an adorable cameo in Khloe Kardashian's new makeup tutorial video"
    },
    {
        "source": "https://www.cnn.com/2019/07/12/entertainment/heidi-klum-tom-kaulitz/",
        "text": "Heidi Klum is apparently the latest celeb to get married and not tell us"
    },
    {
        "source": "https://www.cnn.com/2019/07/19/investing/dow-stock-market-today/",
        "text": "Stocks end lower as geopolitical fears rise. The Dow and US markets closed lower on Friday, as geopolitical worries overshadowed the hopes of interest rate cuts by the Federal Reserve."
    },
    {
        "source": "https://www.cnn.com/2019/07/19/health/astronaut-exercise-iv-faint-scn/",
        "text": "Exercise in space keeps astronauts from fainting when they return to Earth, study says."
    },
]

print("=== Prédictions sur articles CNN (zero-shot) ===\n")
for article in cnn_articles:
    preds = predict_with_ollama(article['text'], CATEGORIES, top_k=TOP_K, mode="zero_shot")
    print(f"  Article : {article['text'][:100]}...")
    print(f"  Top-{TOP_K}  : {preds}")
    print()

---
## 10. Bonus — Tester différents modèles Ollama

In [ ]:
# Décommenter et adapter pour tester plusieurs modèles
# Assurez-vous d'avoir pull les modèles avec `ollama pull <model>`

MODELS_TO_TEST = [
    # "llama3.1:8b",
    # "mistral",
    # "gemma2:9b",
    # "phi3",
]

if MODELS_TO_TEST:
    # Sous-échantillon plus petit pour le benchmark multi-modèles
    benchmark_data = eval_data.sample(n=min(50, len(eval_data)), random_state=123).reset_index(drop=True)
    
    all_results = []
    for model_name in MODELS_TO_TEST:
        print(f"\n--- Test du modèle : {model_name} ---")
        try:
            res, _ = run_evaluation(
                eval_df=benchmark_data,
                categories=CATEGORIES,
                model=model_name,
                top_k=TOP_K,
                mode="zero_shot"
            )
            all_results.append(res)
            print(f"  Accuracy@{TOP_K}: {res['accuracy_at_k']:.3f} | MRR: {res['mrr_at_k']:.3f}")
        except Exception as e:
            print(f"  Erreur : {e}")
    
    if all_results:
        df_benchmark = pd.DataFrame(all_results)
        print("\n=== Comparaison des modèles ===")
        display(df_benchmark)
else:
    print("Ajoutez des modèles à MODELS_TO_TEST pour lancer le benchmark multi-modèles.")

---
## 11. Sauvegarde des résultats

In [ ]:
# Sauvegarder les résultats d'évaluation
output_path = r"C:\Users\jfurs\Pythonn\OpenClassrooms\DS\P8\high_end_ollama_solution\results\llm_evaluation_results.json"

import os
os.makedirs(os.path.dirname(output_path), exist_ok=True)

results_to_save = {
    "zero_shot": results_zs,
    "few_shot": results_fs,
    "config": {
        "model": OLLAMA_MODEL,
        "eval_sample_size": EVAL_SAMPLE_SIZE,
        "top_k": TOP_K,
        "n_categories": len(CATEGORIES),
    }
}

with open(output_path, 'w') as f:
    json.dump(results_to_save, f, indent=2)

print(f"Résultats sauvegardés dans {output_path}")

---
## Résumé

| Aspect | Notebook NLP (référence) | Notebook LLM (Ollama) |
|--------|--------------------------|------------------------|
| **Features** | TF-IDF / Binary / Counts | Texte brut envoyé au LLM |
| **Modèle** | Logistic Regression (sklearn) | LLM local (Ollama) |
| **Entraînement** | Nécessaire (fit sur train set) | Aucun (zero-shot) ou quelques exemples (few-shot) |
| **Vitesse** | Très rapide (~secondes) | Lent (~1-5s par article) |
| **Flexibilité** | Fixé aux catégories du train | Peut classifier vers de nouvelles catégories |
| **Évaluation** | Accuracy + MRR@k | Accuracy + MRR@k (identique) |